# nflbowl — Colab GPU quickstart

Trains the primary 12F encoder (F4, physics-informed contrastive + matched-state alignment) on a Colab GPU, evaluates it and saves every artifact to Google Drive.

**Before you start**
1. *Runtime → Change runtime type → GPU* (a T4 is enough).
2. Build the data bundle on the research machine: `python -m nflbowl.data.builder --repo-root . --output bundles/m12f_v1 --zip` (or `01_data_preparation.ipynb`).
3. Upload **either** `bundles/m12f_v1.zip` to `MyDrive/NFLBowl/` (one file; extracted by Cell 4) **or** the whole folder to `MyDrive/NFLBowl/bundles/m12f_v1/`.
4. A private repository needs a token in Cell 1 (see `docs/nflbowl/COLAB.md`).

**After a disconnect / runtime reset:** run Cells 1–5 again with the SAME `name=`; `exp.train(..., resume=True)` continues from the last epoch checkpoint on Drive, and finished phases are reused (`reuse=True`).

In [ ]:
# Cell 1 — install (keeps Colab's own PyTorch/CUDA build: torch is not a dependency, nothing torch-related is installed)
# Line 1 pulls the light dependencies; line 2 forces the LATEST commit of main over any cached build of the same version.
# If nflbowl was already imported in this runtime: Runtime → Restart session, then continue from Cell 2.
%pip install -q "git+https://github.com/mybenkhadda/NFL_bowl.git@main"
%pip install -q --upgrade --force-reinstall --no-deps "git+https://github.com/mybenkhadda/NFL_bowl.git@main"

In [ ]:
# Cell 2 — version + GPU detection
import nflbowl
from nflbdb.research.advanced_movement.m12 import shared_movement_encoder as se
from nflbowl import Experiment

print("nflbowl", nflbowl.__version__)
# 0.2.1+: contrastive / MMD losses run in fp32 under fp16 autocast (no more "c10::Half without overflow"); no patching needed
assert hasattr(se, "_fp32"), "old nflbowl build still imported: Runtime → Restart session and rerun Cell 1"

info = Experiment.device_info()
if not info.get("cuda_available"):
    print("WARNING: no GPU. Runtime → Change runtime type → GPU (a T4 is enough); on CPU F4 takes hours.")

In [ ]:
# Cell 3 — persistent storage
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Cell 4 — locate the bundle BEFORE creating the experiment (extracts the uploaded ZIP once; validates the manifest)
from pathlib import Path

from nflbowl.data.builder import extract_bundle_zip
from nflbowl.data.loading import open_bundle

DRIVE = Path("/content/drive/MyDrive/NFLBowl")          # output_dir: checkpoints, embeddings, CSVs, figures
# the first candidate holding a bundle.json is used; put your own location first if it differs
BUNDLE_CANDIDATES = [Path("/content/drive/MyDrive/nfl-big-data-bowl-2027/bundles/m12f_v1"), DRIVE / "bundles" / "m12f_v1"]
ZIP_CANDIDATES = [DRIVE / "m12f_v1.zip", DRIVE / "bundles" / "m12f_v1.zip"]
VERIFY_ONCE = False          # True: re-hash every file (≈ 1 min on Drive); do it once after an upload

BUNDLE_DIR = next((p for p in BUNDLE_CANDIDATES if (p / "bundle.json").is_file()), DRIVE / "bundles" / "m12f_v1")
if not (BUNDLE_DIR / "bundle.json").is_file():
    z = next((p for p in ZIP_CANDIDATES if p.is_file()), None)
    if z is None:
        raise FileNotFoundError(
            f"No bundle at {[str(p) for p in BUNDLE_CANDIDATES]} and no ZIP at {[str(p) for p in ZIP_CANDIDATES]}.\n"
            "Build it on the research machine:  python -m nflbowl.data.builder --repo-root . --output bundles/m12f_v1 --zip\n"
            "then upload bundles/m12f_v1.zip to MyDrive/NFLBowl/ (or the whole folder to MyDrive/NFLBowl/bundles/m12f_v1/).")
    print(f"extracting {z} → {BUNDLE_DIR.parent}")
    BUNDLE_DIR = extract_bundle_zip(z, BUNDLE_DIR.parent, verify=True)
print("bundle:", BUNDLE_DIR)

summary = open_bundle(BUNDLE_DIR, verify=VERIFY_ONCE, outer_folds=5).summary()
{k: summary[k] for k in ("n_sequences", "n_combine", "n_nfl", "n_athletes", "frames", "channels", "has_athletes", "has_tcn",
                         "run_mode", "store_key", "fingerprint", "integrity")}

In [ ]:
# Cell 5 — initialise (re-running this with the same name REOPENS the experiment and its Drive artifacts)
exp = Experiment(
    data_dir=str(BUNDLE_DIR),                   # staged to /content/nflbowl_stage by the colab_gpu preset (fast local reads)
    output_dir=str(DRIVE),                      # checkpoints, embeddings, CSVs, figures persist on Drive
    config="colab_gpu",
    device="auto",
    name="f4_main",
)
exp

In [ ]:
# Cell 6 — prepare data (stage to local disk, validate, cache derived targets + F0 features on Drive)
exp.prepare()

In [ ]:
# Cell 7 — train the primary encoder (resumes from the last Drive epoch checkpoint if interrupted; a finished run is restored)
# A nonfinite loss stops training with FloatingPointError naming the loss term / epoch / batch; saved checkpoints stay valid.
training_table = exp.train("F4", resume=True)
display(training_table)

In [ ]:
# Cell 8 — embeddings + evaluation (reuse=True: saved embeddings / records are loaded, not recomputed)
exp.embed("F4", reuse=True)
exp.evaluate("F0", level="standard", reuse=True)
exp.evaluate("random", level="standard", reuse=True)
exp.evaluate("F4", level="standard", reuse=True)
exp.compare(level="standard")

In [ ]:
# Cell 9 — figures, report, export (all written under MyDrive/NFLBowl/experiments/f4_main)
figs = exp.plot()
print(exp.report())
print(exp.export())

## Reconnecting in a new session

```python
%pip install -q "git+https://github.com/mybenkhadda/NFL_bowl.git@main"
%pip install -q --upgrade --force-reinstall --no-deps "git+https://github.com/mybenkhadda/NFL_bowl.git@main"
from google.colab import drive; drive.mount("/content/drive")
from nflbowl import Experiment
exp = Experiment.open("/content/drive/MyDrive/NFLBowl/experiments/f4_main",
                      data_dir="/content/drive/MyDrive/nfl-big-data-bowl-2027/bundles/m12f_v1",   # your bundle location
                      runtime={"stage_dir": "/content/nflbowl_stage"})
exp.prepare()                 # re-stages the bundle; derived tables come from the Drive cache
exp.resume()                  # continues any training job left 'running'
exp.embed("F4", reuse=True)   # loads saved embeddings — no recomputation
exp.evaluate("F4", level="standard", reuse=True)
```

No runtime patching of library internals is needed: since nflbowl 0.2.1 the contrastive losses are computed in fp32 under fp16 autocast (see `docs/nflbowl/COLAB.md` § Mixed precision).

`bundle.json not found`? Run Cell 4: it names the missing piece (Drive not mounted, ZIP not extracted, folder uploaded one level too high, incomplete upload). See `docs/nflbowl/BUNDLE_TROUBLESHOOTING.md`.